# 22장 · 벡터 거리와 Chroma·FAISS 검색

벡터 검색은 질문과 문서를 같은 좌표 체계로 나타내고 거리가 가까운 문서를 찾습니다. 여기서는 단어의 포함 여부로 좌표를 만들어 거리를 직접 검산합니다.

‘대출 기간은?’이라는 질문과 세 안내 문장을 숫자 목록으로 바꾸면 어떤 문장이 가장 가까울까요? 같은 벡터를 직접 비교한 뒤 Chroma와 FAISS에 똑같이 넣어 검색 결과를 읽습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 faiss-cpu 1.15.1, numpy 2.5.4입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 어휘 벡터와 제곱 L2 거리

먼저 ‘대출, 휴관, 반납’ 세 단어가 있으면 1, 없으면 0으로 표시합니다. 이 수업용 표현은 눈으로 계산을 따라가려는 것입니다. 의미가 비슷한 문장을 학습한 임베딩 모델의 결과와는 다릅니다.

**질의 벡터와 문서 간 제곱거리**　대출 질의와 대출 문서의 제곱거리는 0, 휴관 문서와는 2입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter22/index.html#practice-space)


### 셀 01 · 어휘 순서와 문서 벡터

20장에서 등록한 세 원문의 ID, 본문, 출처, 쪽수를 그대로 사용합니다.


In [1]:
documents = [
  {"id":"library-1","text":"도서 대출 기간은 14일입니다.","source":"가상 도서관 이용 안내","page":1},
  {"id":"library-2","text":"도서관은 월요일에 휴관합니다.","source":"가상 도서관 이용 안내","page":2},
  {"id":"library-3","text":"무인 반납함은 24시간 이용할 수 있습니다.","source":"가상 도서관 이용 안내","page":3}
]

terms = ("대출", "휴관", "반납")

def teaching_vector(text):
    return [float(term in text) for term in terms]

vectors = [teaching_vector(d["text"]) for d in documents]
print(vectors[0])

[1.0, 0.0, 0.0]


library-1은 [1.0, 0.0, 0.0]입니다. 첫 숫자는 대출이 있다는 뜻입니다. 저장할 때와 검색할 때 단어 순서가 달라지면 숫자가 셋이어도 비교가 틀어집니다.


### 셀 02 · 제곱 L2 거리 계산

질문 벡터와 휴관 안내 벡터에서 같은 위치끼리 빼고 제곱해 더합니다.


In [2]:
query = teaching_vector("대출 기간은?")
closed_vector = vectors[1]
print(sum((a - b) ** 2 for a, b in zip(query, closed_vector)))

2.0


2.0입니다. 첫 두 좌표가 각각 1만큼 다르고 마지막 좌표는 같아서 1+1+0이 됩니다. 이 거리는 답변의 정답 확률이 아닙니다.


### 과제 03 · 질의 벡터와 문서 거리

같은 질문을 첫 문서와 비교해 제곱 거리 합을 구하세요.


In [3]:
# query와 vectors[0]의 같은 좌표끼리 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(sum((a - b) ** 2 for a, b in zip(query, vectors[0])))</code></pre>
<p>실행 결과</p><pre><code>0.0</code></pre>
<p>0.0은 이 표현에서 두 벡터가 같다는 뜻입니다. ‘대출 가능 권수’처럼 의도가 다른 질문도 같은 벡터가 될 수 있습니다.</p>
</details>


## Chroma 컬렉션의 저장과 조회

수업용 커널에 chromadb가 있으면 새 임시 폴더에 컬렉션을 만듭니다. 이번 노트북에서는 같은 세 문서와 모델 이름을 사용하되, 임시 컬렉션은 다른 장의 저장소를 덮어쓰지 않습니다. 아래 두 셀은 실제 Chroma를 쓰는 선택 경로입니다.


### 셀 04 · Chroma에 본문과 메타데이터 저장

임베딩 값을 직접 주므로 외부 임베딩 모델을 내려받지 않습니다.


In [4]:
from importlib.util import find_spec
from tempfile import TemporaryDirectory
has_chroma = find_spec("chromadb") is not None
if has_chroma:
    import chromadb
    chroma_folder = TemporaryDirectory()
    client = chromadb.PersistentClient(path=chroma_folder.name)
    collection = client.create_collection("library_course_v1_e1", embedding_function=None,
                                         metadata={"model": "keyword-presence-v1"})
    collection.upsert(ids=[d["id"] for d in documents], embeddings=vectors,
                      documents=[d["text"] for d in documents],
                      metadatas=[{"source": d["source"], "page": d["page"]} for d in documents])
    print(collection.count())
else:
    print("실제 저장 실습은 chromadb가 설치된 커널에서 실행하세요.")

실제 저장 실습은 chromadb가 설치된 커널에서 실행하세요.


실제 컬렉션에 세 건을 넣으면 3이 나옵니다. 각 문서의 위치와 벡터가 같은 순서로 들어갔습니다. 전체 인덱스 인수인계 시에는 이름 외에도 차원·거리·원문 버전을 대조합니다.


### 셀 05 · 컬렉션 재조회와 결과 ID

앞 셀에서 만든 같은 경로와 이름을 지정합니다. 자료를 다시 적재하지 않고 조회합니다.


In [5]:
if has_chroma:
    reopened = chromadb.PersistentClient(path=chroma_folder.name).get_collection(
        "library_course_v1_e1", embedding_function=None)
    found = reopened.query(query_embeddings=[query], n_results=1, include=["metadatas", "distances"])
    print(found["ids"][0][0], found["metadatas"][0][0]["page"])
else:
    print("Chroma 재열기는 실제 저장 셀을 실행한 뒤 확인하세요.")

Chroma 재열기는 실제 저장 셀을 실행한 뒤 확인하세요.


Chroma가 있으면 library-1, 1쪽이 나옵니다. ids의 첫 [0]은 첫 질문, 다음 [0]은 그 질문의 1순위입니다.


## FAISS 인덱스와 문서 ID 매핑

이번에는 같은 숫자들을 FAISS의 L2 인덱스에 넣습니다. 검색 결과가 행 번호로 오므로 원래 documents의 같은 위치에서 ID를 꺼냅니다.


#### 동일 질의 벡터의 Chroma·FAISS 검색 비교

<table style="width:100%;border-collapse:collapse"><thead><tr><th scope="col">저장소</th><th scope="col">노트북에 저장된 출력</th><th scope="col">내 커널에서 확인할 값</th></tr></thead><tbody><tr><td>Chroma · 셀 04~05</td><td>패키지 없음 · 저장과 재열기 미실행</td><td>다시 열고 첫 검색 문서가 library-1인지 확인</td></tr><tr><td>FAISS · 셀 06</td><td>library-1 · 거리 0.0</td><td>같은 질의 벡터로 ID와 거리 대조</td></tr></tbody></table>


### 셀 06 · FAISS 거리 검색

faiss와 numpy가 있는 커널에서 실제 인덱스를 메모리에 만듭니다.

같은 대출 질문으로 비교하세요. 제공된 Chroma 셀에는 설치 안내만 저장되어 있어 그 실행의 검색 ID는 미확인입니다. FAISS 셀에는 library-1과 거리 0.0이 저장되어 있습니다. Chroma를 실행했다면 ids[0][0]을 읽어 이 ID와 대조하세요.


In [6]:
if find_spec("faiss") is not None and find_spec("numpy") is not None:
    import faiss
    import numpy as np
    index = faiss.IndexFlatL2(3)
    index.add(np.array(vectors, dtype="float32"))
    distances, positions = index.search(np.array([query], dtype="float32"), 1)
    print(documents[int(positions[0][0])]["id"], float(distances[0][0]))
else:
    print("FAISS 실습에는 faiss와 numpy가 설치된 수업 커널이 필요합니다.")

library-1 0.0


실제 조회를 하면 library-1과 0.0이 나옵니다. positions는 문자열 ID가 아니므로 문서 목록의 순서를 바꾸면 연결도 다시 맞춰야 합니다. Chroma 실습은 수업 커널에 chromadb가 있는지 먼저 확인하세요. 새로 연 컬렉션에서 첫 ID를 읽었을 때만 FAISS에서 나온 첫 ID와 한 줄에 나란히 적으세요.


## 돌아보기

‘주차 요금’의 벡터가 [0, 0, 0]이라면 자료가 없는 질문에도 가까운 순서가 나올 수 있습니다. 다음 장의 검색 함수에는 교육용 영벡터를 빈 결과로 다룬다는 규칙을 함께 넘깁니다.
